# Speech-Based Cognitive Screening — Notebook 05
## Clean-Audio Re-extraction · isolate the patient, then measure again

**Team:** A. R. Nitesh Kumar (23BAI0108) · Harshini J. S. (23BAI0166) · Hari Prasath U. (23BAI0038)  


**Why this notebook exists.**  
In Notebook 04 the acoustic features were computed over the **whole recording** — which includes the
*interviewer's* voice and silence, not just the patient. We measured the contamination on the real
files: the patient speaks only **~45–75%** of each recording. That is why audio underperformed text
(the text features used the clean `*PAR:` tier; the audio didn't).

This notebook fixes that. It uses the **transcript timestamps** — the corpus's own gold-standard
segmentation — to keep **only the patient's speech**, re-extracts the acoustic features, and re-runs
the comparison. This is the honest test of whether voice adds diagnostic value.

> **Why timestamps, not a diarization model?** The `.cha` files already tell us exactly when the
> patient speaks (millisecond marks on every `*PAR:` line). Those boundaries are ground truth. A
> model like pyannote would only *estimate* them and could add errors. Using the corpus's own
> segmentation is cleaner, reproducible, needs no token, and is the stronger choice to defend.

### Upload to this Colab session first: `Pitt.zip`, `cookies.txt`, `pitt_cookie_features.csv`.
Then **Runtime → Run all**. (If you still have the `audio/` folder from Notebook 04 in this session,
Cell 2 will reuse it and skip re-downloading.)


## 0 · Setup

In [ ]:
!pip install librosa praat-parselmouth pylangacq pandas pyarrow scikit-learn tqdm requests --quiet
import os, glob, re, requests, numpy as np, pandas as pd
import librosa, parselmouth, pylangacq
from parselmouth.praat import call
from tqdm import tqdm
print("setup OK")

## 1 · Make sure the audio is present

If the `audio/` folder from Notebook 04 is still in this session, we reuse it. Otherwise, this cell
re-downloads with your cookie (same working method as Notebook 04).


In [ ]:
BASE = "https://media.talkbank.org/dementia/English/Pitt"

# unzip transcripts (needed for timestamps AND the file list)
if glob.glob("Pitt.zip") and not glob.glob("**/Dementia/cookie/*.cha", recursive=True):
    os.system("unzip -o Pitt.zip -d . > /dev/null 2>&1")

have = glob.glob("audio/**/*.mp3", recursive=True)
if len(have) > 100:
    print(f"reusing {len(have)} MP3s already in session — skipping download")
else:
    assert os.path.exists("cookies.txt"), "Upload cookies.txt (from your logged-in TalkBank tab)."
    sess = requests.Session()
    for line in open("cookies.txt"):
        if line.startswith("#") or not line.strip(): continue
        p = line.rstrip("\n").split("\t")
        if len(p) >= 7: sess.cookies.set(p[5], p[6], domain=".talkbank.org")

    ids = []
    for grp in ["Control","Dementia"]:
        for f in glob.glob(f"**/{grp}/cookie/*.cha", recursive=True):
            ids.append((grp, os.path.basename(f)[:-4]))
    ids = sorted(set(ids))
    os.makedirs("audio", exist_ok=True)
    ok = 0
    for grp, fid in tqdm(ids, desc="download"):
        out = f"audio/{grp}/{fid}.mp3"; os.makedirs(os.path.dirname(out), exist_ok=True)
        if os.path.exists(out) and os.path.getsize(out) > 5000: ok += 1; continue
        r = sess.get(f"{BASE}/{grp}/cookie/{fid}.mp3", stream=True, timeout=60,
                     headers={"Range": "bytes=0-"})
        content = b"".join(r.iter_content(chunk_size=8192))
        if len(content) > 5000 and content[:1] != b'<':
            open(out, "wb").write(content); ok += 1
    print("downloaded", ok, "MP3s")

## 2 · Map each transcript's patient-speech timestamps

For every `.cha` file we read the millisecond marks on the `*PAR:` utterances — these are the exact
windows where the patient (not the interviewer) is speaking.


In [ ]:
def par_segments(cha_path):
    """Return list of (start_ms, end_ms) for PAR utterances."""
    chat = pylangacq.read_chat(cha_path)
    segs = []
    for u in chat.utterances():
        if u.participant == "PAR" and u.time_marks:
            s, e = u.time_marks
            if e > s: segs.append((s, e))
    return segs

# build a lookup: file id -> segments
seg_map = {}
for grp in ["Control", "Dementia"]:
    for f in glob.glob(f"**/{grp}/cookie/*.cha", recursive=True):
        fid = os.path.basename(f)[:-4]
        seg_map[fid] = par_segments(f)

# quick contamination report
fracs = []
for fid, segs in seg_map.items():
    if segs:
        par = sum(e-s for s,e in segs)
        span = max(e for _,e in segs) - min(s for s,_ in segs)
        if span > 0: fracs.append(par/span)
print(f"transcripts with timestamps: {sum(len(v)>0 for v in seg_map.values())}/{len(seg_map)}")
print(f"patient speaks on average {np.mean(fracs)*100:.0f}% of each recording "
      f"(the other {100-np.mean(fracs)*100:.0f}% was interviewer + silence we now remove)")

## 3 · Re-extract acoustics from patient-only audio

Same feature set as Notebook 04, but computed on the **concatenated patient segments** instead of the
whole file. Takes a few minutes.


In [ ]:
def acoustic_features(y, sr):
    f = {}
    try:
        snd = parselmouth.Sound(y.astype(np.float64), sampling_frequency=sr)
        pitch = snd.to_pitch(); f0v = pitch.selected_array['frequency']; f0v = f0v[f0v>0]
        f["f0_mean"]=float(np.mean(f0v)) if len(f0v) else np.nan
        f["f0_std"] =float(np.std(f0v))  if len(f0v) else np.nan
        f["f0_range"]=float(np.ptp(f0v)) if len(f0v) else np.nan
        pp = call(snd, "To PointProcess (periodic, cc)", 75, 500)
        f["jitter_local"]=call(pp,"Get jitter (local)",0,0,1e-4,0.02,1.3)
        f["shimmer_local"]=call([snd,pp],"Get shimmer (local)",0,0,1e-4,0.02,1.3,1.6)
        harm=call(snd,"To Harmonicity (cc)",0.01,75,0.1,1.0); f["hnr"]=call(harm,"Get mean",0,0)
    except Exception:
        for kk in ["f0_mean","f0_std","f0_range","jitter_local","shimmer_local","hnr"]:
            f.setdefault(kk, np.nan)
    f["spec_centroid"]=float(np.mean(librosa.feature.spectral_centroid(y=y,sr=sr)))
    f["spec_bandwidth"]=float(np.mean(librosa.feature.spectral_bandwidth(y=y,sr=sr)))
    f["zcr"]=float(np.mean(librosa.feature.zero_crossing_rate(y)))
    f["rms_energy"]=float(np.mean(librosa.feature.rms(y=y)))
    mfcc=librosa.feature.mfcc(y=y,sr=sr,n_mfcc=13)
    for i in range(13): f[f"mfcc{i+1}"]=float(np.mean(mfcc[i]))
    return f

def slice_par(y, sr, segs_ms):
    chunks=[]
    for s,e in segs_ms:
        a=int(s/1000*sr); b=int(e/1000*sr)
        if 0<=a<b<=len(y): chunks.append(y[a:b])
    return np.concatenate(chunks) if chunks else np.array([],dtype=np.float32)

rows=[]
mp3s = sorted(glob.glob("audio/**/*.mp3", recursive=True))
for path in tqdm(mp3s, desc="clean-extract"):
    fid = os.path.basename(path)[:-4]
    segs = seg_map.get(fid, [])
    try:
        y, sr = librosa.load(path, sr=16000, mono=True)
        y_par = slice_par(y, sr, segs) if segs else y   # fall back to full if no segs
        if len(y_par) < sr*0.5:                          # <0.5s of speech -> skip acoustics
            continue
        feats = acoustic_features(y_par, sr)
        feats["file"] = fid
        feats["par_speech_s"] = len(y_par)/sr
        rows.append(feats)
    except Exception as e:
        print("skip", fid, e)

audio_clean = pd.DataFrame(rows)
audio_clean.to_parquet("pitt_cookie_audio_CLEAN.parquet", index=False)
print(f"\nclean acoustic features for {len(audio_clean)} recordings")
audio_clean[["file","f0_mean","jitter_local","hnr","par_speech_s"]].head()

## 4 · Merge and compare — did cleaning help?

Merge the **clean** acoustic features with the text features, then run text vs audio vs fused again.
The key comparison: does clean audio beat the contaminated audio from Notebook 04 (which scored ~0.69)?


In [ ]:
tcand = [c for c in glob.glob("**/pitt_cookie_features.*", recursive=True)
         if "AUDIO" not in c and "CLEAN" not in c and "audio" not in c]
if not tcand:
    from google.colab import files
    print("Upload pitt_cookie_features.csv from Notebook 01 …"); files.upload()
    tcand = [c for c in glob.glob("**/pitt_cookie_features.*", recursive=True)
             if "AUDIO" not in c and "CLEAN" not in c and "audio" not in c]
tpath = ([c for c in tcand if c.endswith(".parquet")] or tcand)[0]
text_df = pd.read_parquet(tpath) if tpath.endswith(".parquet") else pd.read_csv(tpath)

merged = text_df.merge(audio_clean, on="file", how="left")
merged.to_csv("pitt_cookie_features_CLEAN.csv", index=False)
merged.to_parquet("pitt_cookie_features_CLEAN.parquet", index=False)

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

d = merged[merged.dx.isin(["Control","ProbableAD"])].copy()
d["label"] = (d.dx=="ProbableAD").astype(int)
TEXT = ["mattr","pronoun_noun_ratio","mlu_words","lexical_density","filled_pause_rate",
        "speech_rate_wpm","long_pause_ratio","mean_pause_ms","ttr","noun_verb_ratio"]
AUDIO = ["f0_mean","f0_std","f0_range","jitter_local","shimmer_local","hnr",
         "spec_centroid","spec_bandwidth","zcr","rms_energy"] + [f"mfcc{i}" for i in range(1,14)]
TEXT=[c for c in TEXT if c in d.columns]; AUDIO=[c for c in AUDIO if c in d.columns]

def cv_auc(cols):
    sub=d.dropna(subset=cols, how="all")
    X=sub[cols].values; y=sub.label.values; g=sub.participant_id.values
    sgkf=StratifiedGroupKFold(5,shuffle=True,random_state=42); a=[]
    for tr,te in sgkf.split(X,y,g):
        assert not(set(g[tr])&set(g[te]))
        clf=make_pipeline(SimpleImputer(),StandardScaler(),
                          RandomForestClassifier(n_estimators=300,random_state=0))
        clf.fit(X[tr],y[tr]); a.append(roc_auc_score(y[te],clf.predict_proba(X[te])[:,1]))
    return np.mean(a),np.std(a)

t=cv_auc(TEXT); a=cv_auc(AUDIO); f=cv_auc(TEXT+AUDIO)
print(f"Text only         AUC {t[0]:.3f} ± {t[1]:.3f}")
print(f"Audio (CLEAN)     AUC {a[0]:.3f} ± {a[1]:.3f}   <- compare to 0.69 contaminated")
print(f"Text + Audio      AUC {f[0]:.3f} ± {f[1]:.3f}")
print(f"\n>>> Clean audio vs text: {a[0]-t[0]:+.3f} · Fusion vs text: {f[0]-t[0]:+.3f} <<<")

## 5 · Read the result honestly

- **If clean audio > 0.69 (the contaminated score):** cleaning helped — the interviewer really was
  diluting the signal. Report both numbers; the improvement is itself a methodological finding.
- **If fusion now beats text:** voice genuinely adds diagnostic value once isolated — your original
  intuition confirmed.
- **If text still wins even with clean audio:** that is a strong, honest result — for AD detection
  from this task, *what* is said dominates *how* it sounds. Well-supported by the literature.

Either way, `pitt_cookie_features_CLEAN.csv` is now your best table.

### Next in the framework
1. **Feed `pitt_cookie_features_CLEAN.csv` into Notebooks 02 & 03** (add acoustic columns to
   `CONCEPTS`) — interpretability cost and XAI faithfulness, now including clean voice.
2. **Grad-CAM spectrogram CNN** — the imaging-style visual explanation (uses your medical-imaging
   background); train on patient-only spectrograms.
3. **Early-exit experiment** — "how little speech is enough", the other novel pillar.

> Report every metric as **mean ± std over folds/seeds**.
